# Pipeline de transformation des données

**Source** : `housing.csv` (données brutes) → **Sortie** : `amputed_normalized_housing_data.csv`

| Étape | Opération | Détail |
|-------|-----------|--------|
| 1 | Imputation KNN | `total_bedrooms` — 207 NaN comblés par $k=5$ voisins pondérés par distance |
| 2 | Encodage ordinal | `ocean_proximity` → 0–4 |
| 3 | Nettoyage outliers | Suppression prix plafonnés (500 001$) + IQR sur features numériques |
| 4 | Normalisation Z-Score | $z = \frac{x - \mu}{\sigma}$ sur toutes les features sauf `median_house_value` (cible) |

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

housing = pd.read_csv('./datasets/housing.csv')
print(f"Shape : {housing.shape}")
print(f"Valeurs manquantes : total_bedrooms → {housing['total_bedrooms'].isnull().sum()}")

Shape : (20640, 10)
Valeurs manquantes : total_bedrooms → 207


## 1. Imputation KNN ($k=5$, pondération par distance)

$$\hat{x}_i = \frac{\sum_{j \in \mathcal{N}_k(i)} w_j \cdot x_j}{\sum_{j \in \mathcal{N}_k(i)} w_j}, \quad w_j = \frac{1}{d(i,j)}$$

On standardise temporairement avant KNN pour que la distance euclidienne soit cohérente entre colonnes.

In [2]:
# Encodage temporaire de ocean_proximity pour KNN (numérique requis)
ocean_mapping = {'<1H OCEAN': 0, 'INLAND': 1, 'ISLAND': 2, 'NEAR BAY': 3, 'NEAR OCEAN': 4}
housing['ocean_proximity'] = housing['ocean_proximity'].map(ocean_mapping)

# Standardisation temporaire → KNN → dé-standardisation
num_cols = housing.columns.tolist()
means, stds = housing[num_cols].mean(), housing[num_cols].std()
X_scaled = (housing[num_cols] - means) / stds

imputer = KNNImputer(n_neighbors=5, weights='distance')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_scaled) * stds.values + means.values,
    columns=num_cols, index=housing.index
)
housing['total_bedrooms'] = X_imputed['total_bedrooms'].round(0)

print(f"Valeurs manquantes après imputation : {housing.isnull().sum().sum()}")

Valeurs manquantes après imputation : 0


## 2. Nettoyage des outliers

**2a.** Suppression des prix plafonnés (`median_house_value == 500001`) — données censurées, pas des prix réels.

**2b.** Suppression des outliers IQR sur les features numériques (sauf `ocean_proximity` et `median_house_value`) :

$$\text{outlier si } x < Q_1 - 1.5 \times IQR \text{ ou } x > Q_3 + 1.5 \times IQR$$

In [3]:
n_before = len(housing)

# 2a. Suppression des prix plafonnés
capped = (housing['median_house_value'] == 500001).sum()
housing = housing[housing['median_house_value'] != 500001]
print(f"Prix plafonnés supprimés : {capped}")

# 2b. Suppression des outliers IQR sur les features numériques
iqr_cols = [c for c in housing.columns if c not in ('ocean_proximity', 'median_house_value')]
mask = pd.Series(True, index=housing.index)

for col in iqr_cols:
    q1 = housing[col].quantile(0.25)
    q3 = housing[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    col_mask = (housing[col] >= lower) & (housing[col] <= upper)
    n_out = (~col_mask).sum()
    if n_out > 0:
        print(f"  {col:>20s} : {n_out} outliers (bornes [{lower:.1f}, {upper:.1f}])")
    mask &= col_mask

housing = housing[mask].reset_index(drop=True)
n_after = len(housing)
print(f"\nLignes supprimées : {n_before - n_after} ({(n_before - n_after)/n_before*100:.1f}%)")
print(f"Lignes restantes  : {n_after}")

Prix plafonnés supprimés : 965
           total_rooms : 1251 outliers (bornes [-1085.0, 5643.0])
        total_bedrooms : 1227 outliers (bornes [-229.5, 1174.5])
            population : 1126 outliers (bornes [-629.0, 3171.0])
            households : 1165 outliers (bornes [-204.0, 1092.0])
         median_income : 364 outliers (bornes [-0.6, 7.7])

Lignes supprimées : 3039 (14.7%)
Lignes restantes  : 17601


## 3. Normalisation Z-Score (features uniquement)

$$z = \frac{x - \mu}{\sigma}$$

`median_house_value` reste en valeur brute — c'est la cible du perceptron.

In [4]:
target = 'median_house_value'
feature_cols = [c for c in housing.columns if c != target]

scaler = StandardScaler()
housing[feature_cols] = scaler.fit_transform(housing[feature_cols])

print("Features (mean ≈ 0, std ≈ 1) :")
print(housing[feature_cols].describe().loc[['mean', 'std']].round(4))
print(f"\nmedian_house_value (non transformé) : mean={housing[target].mean():.0f}, std={housing[target].std():.0f}")

Features (mean ≈ 0, std ≈ 1) :
      longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
mean        0.0       0.0                -0.0          0.0             0.0   
std         1.0       1.0                 1.0          1.0             1.0   

      population  households  median_income  ocean_proximity  
mean         0.0        -0.0           -0.0              0.0  
std          1.0         1.0            1.0              1.0  

median_house_value (non transformé) : mean=187322, std=94874


## 4. Export

In [5]:
output_path = './datasets/amputed_normalized_housing_data.csv'
housing.to_csv(output_path, index=False)
print(f"Exporté : {output_path} — {housing.shape[0]} lignes, {housing.shape[1]} colonnes")
housing.head()

Exporté : ./datasets/amputed_normalized_housing_data.csv — 17601 lignes, 10 colonnes


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-1.315977,0.995320,1.83728,-0.622220,-1.156299,-1.161352,-1.168233,2.620165,352100.0,1.300935
1,-1.320967,0.995320,1.83728,-0.799883,-0.953182,-1.058970,-0.964205,1.477758,341300.0,1.300935
2,-1.320967,0.995320,1.83728,-0.474934,-0.750065,-1.047410,-0.769893,0.206130,342200.0,1.300935
3,-1.320967,0.995320,1.83728,-1.126674,-1.052484,-1.298413,-1.090508,0.341013,269700.0,1.300935
4,-1.320967,0.990705,1.83728,0.360912,0.193300,-0.173855,0.468851,0.073723,299200.0,1.300935


In [6]:
import joblib

# Sauvegarder le scaler pour réutilisation dans predict.py
joblib.dump(scaler, './models/scaler.joblib')
joblib.dump(feature_cols, './models/feature_cols.joblib')
print("Scaler exporté → ./models/scaler.joblib")
print(f"Features scalées : {feature_cols}")

Scaler exporté → ./models/scaler.joblib
Features scalées : ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'ocean_proximity']
